In [17]:
import xarray as xr
import time

In [13]:
def create_input_ds(scen,num_string):
    tasmax_ds = xr.open_dataset("/nesi/project/niwa02986/queenle/data/d4pdf/WNAmerica/raw_inputs/"+scen+"/TA/TA_Aday_d4PDF_"+scen+"_m"+num_string+"_1951-2021.nc")
    rhmin_ds = xr.open_dataset("/nesi/project/niwa02986/queenle/data/d4pdf/WNAmerica/raw_inputs/"+scen+"/RHA/RHA_Aday_d4PDF_"+scen+"_m"+num_string+"_1951-2021.nc")
    wind_ds = xr.open_dataset("/nesi/project/niwa02986/queenle/data/d4pdf/WNAmerica/raw_inputs/"+scen+"/WIND/WIND_Aday_d4PDF_"+scen+"_m"+num_string+"_1951-2021.nc")
    pr_ds = xr.open_dataset("/nesi/project/niwa02986/queenle/data/d4pdf/WNAmerica/raw_inputs/"+scen+"/PRECIPI/PRECIPI_Aday_d4PDF_"+scen+"_m"+num_string+"_1951-2021.nc")

    # merge variables into one dataset
    FWI_inputs = xr.merge([pr_ds.precipi, tasmax_ds.ta, rhmin_ds.rha, wind_ds.wind]).drop('lev')
    FWI_inputs = FWI_inputs.where(lsm.sftlf > 0)

    # temp K to C
    FWI_inputs["ta"] = FWI_inputs["ta"] - 273.15 # convert temp K to C
    # wind m/s to km/h
    FWI_inputs["wind"] = (FWI_inputs["wind"] * 60 * 60)/1000
    # precip kg/m2/s to mm/hr
    FWI_inputs["precipi"] = FWI_inputs["precipi"]*(60*60*24)

    #FWI_inputs.to_netcdf("/nesi/project/niwa02986/queenle/data/d4pdf/WNAmerica/FWI/inputs/"+scen+"/FWI_inputs_"+scen+"_m"+num_string+"_1951-2021.nc")


In [18]:
ex_da = xr.open_dataset("/nesi/project/niwa02986/queenle/data/d4pdf/WNAmerica/raw_inputs/HPB/TA/TA_Aday_d4PDF_HPB_m001_1951-2021.nc")
lons = ex_da.lon.data
lats = ex_da.lat.data
ex_da.close()

In [20]:
lons

array([222.75  , 223.3125, 223.875 , 224.4375, 225.    , 225.5625,
       226.125 , 226.6875, 227.25  , 227.8125, 228.375 , 228.9375,
       229.5   , 230.0625, 230.625 , 231.1875, 231.75  , 232.3125,
       232.875 , 233.4375, 234.    , 234.5625, 235.125 , 235.6875,
       236.25  , 236.8125, 237.375 , 237.9375, 238.5   , 239.0625,
       239.625 , 240.1875, 240.75  , 241.3125, 241.875 , 242.4375,
       243.    , 243.5625, 244.125 , 244.6875, 245.25  , 245.8125,
       246.375 , 246.9375, 247.5   , 248.0625, 248.625 , 249.1875,
       249.75  , 250.3125, 250.875 , 251.4375, 252.    , 252.5625,
       253.125 , 253.6875, 254.25  , 254.8125, 255.375 ])

In [15]:
'''
Create FWI input netcdf for each ensemble member
'''

ex_da = xr.open_dataset("/nesi/project/niwa02986/queenle/data/d4pdf/WNAmerica/raw_inputs/HPB/TA/TA_Aday_d4PDF_HPB_m001_1951-2021.nc")
lons = ex_da.lon.data
lats = ex_da.lat.data
ex_da.close()

lsm = xr.open_dataset('/nesi/project/niwa02986/queenle/data/d4pdf/sftlf_fx_d4PDF_HPB_m000_000000-000000.nc')
lsm = lsm.isel(time=0).drop('time')
lsm = lsm.interp(lat=lats,lon=lons)

for scen in ['HPB','HPB_NAT']:
    for num in range(1,101):  
        t0 = time.time()
        num_string = '0' + str(num) if num > 9 else '00' + str(num)                                                             
        if num == 100: 
            num_string = '100'  
            
        create_input_ds(scen, num_string)
        
        t1 = time.time()
        total = t1-t0
        print(total)
        
        break
    break

/home/queenle/.local/lib/python3.6/site-packages/xarray/coding/times.py:83: SerializationWarning: Ambiguous reference date string: 1-1-1 00:00:00. The first value is assumed to be the year hence will be padded with zeros to remove the ambiguity (the padded reference date string is: 0001-1-1 00:00:00). To remove this message, remove the ambiguity by padding your reference date strings with zeros.
  warnings.warn(warning_msg, SerializationWarning)


18.769083976745605
